# 03e — Two-Stage Grid · Combine (M × N) + Position Weighted Avg

**역할**: 03c CLF prob × 03d REG pred → 모든 (clf, reg) 조합 unit pred. **Two-Stage 결합 전용**. Stacking 은 외부 노트북에서 별도 처리.

**Step 1 — die-level 곱셈**
- `final_die = clf_prob_die × reg_pred_die`

**Step 2 — die→unit 집계 (2가지)**
- (a) `mean(4 die)` — baseline → `oof|val|test_unit.csv`
- (b) **Position weighted avg** (Optuna 4 weight, Dirichlet 정규화) → `oof|val|test_unit_weighted.csv`

**입력**: `4_output/final/two_stage_grid/clf/{*}/oof|val|test_die.csv` + `reg/{*}/oof|val|test_die.csv`

**출력**: `4_output/final/two_stage_grid/combined/`
- `{clf}_x_{reg}/{oof,val,test}_unit.csv` (mean)
- `{clf}_x_{reg}/{oof,val,test}_unit_weighted.csv` (Optuna weighted)
- `grid_summary.csv`, `weighted_summary.csv`, `combine_meta.json`

**Stacking 은 외부 노트북** 에서 11-base + 24 grid (8 mean + 8 weighted + 8 단독) + path B + NN base 등을 모아 처리.

## 1. 환경 + 자동 탐지

In [1]:
import os, sys, json

%run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all

from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from scipy.optimize import minimize

GRID_ROOT = os.path.join(OUTPUT_DIR, 'final', 'two_stage_grid')
CLF_DIR   = os.path.join(GRID_ROOT, 'clf')
REG_DIR   = os.path.join(GRID_ROOT, 'reg')
OUT_DIR   = os.path.join(GRID_ROOT, 'combined')
os.makedirs(OUT_DIR, exist_ok=True)

REQ_DIE = ['oof_die.csv', 'val_die.csv', 'test_die.csv']

def _list_models(root):
    if not os.path.exists(root):
        return {}
    out = {}
    for name in sorted(os.listdir(root)):
        path = os.path.join(root, name)
        if not os.path.isdir(path):
            continue
        if all(os.path.exists(os.path.join(path, f)) for f in REQ_DIE):
            out[name] = path
    return out

clf_pool = _list_models(CLF_DIR)
reg_pool = _list_models(REG_DIR)

print(f'CLF 가용 ({len(clf_pool)}):  {list(clf_pool)}')
print(f'REG 가용 ({len(reg_pool)}):  {list(reg_pool)}')
print(f'Grid 조합 수: {len(clf_pool) * len(reg_pool)}')

if not clf_pool:
    raise RuntimeError(f'CLF 산출물 없음: {CLF_DIR}/{{model}}/{REQ_DIE}')
if not reg_pool:
    raise RuntimeError(f'REG 산출물 없음: {REG_DIR}/{{model}}/{REQ_DIE}')

setup 완료
CLF 가용 (4):  ['catboost', 'et', 'lgbm', 'xgb']
REG 가용 (5):  ['catboost', 'enet', 'et', 'lgbm', 'xgb']
Grid 조합 수: 20


## 2. CLF/REG die-level 로드 + 정합 검증

각 csv 컬럼: `clf` = `[ufs_serial, run_wf_xy, prob, (health)]`, `reg` = `[ufs_serial, run_wf_xy, pred, (health)]`.

die 단위 정합 보장 (`run_wf_xy` 동일 순서).

In [2]:
from utils.config import DIE_KEY_COL

_, ys = load_all()
y_train = ys['train'].set_index(KEY_COL)[TARGET_COL]
y_val   = ys['validation'].set_index(KEY_COL)[TARGET_COL]
y_test  = ys['test'].set_index(KEY_COL)[TARGET_COL]

def _load_die(path, split, value_col):
    """die csv 로드 → DataFrame [KEY, DIE_KEY, value]."""
    df = pd.read_csv(os.path.join(path, f'{split}_die.csv'))
    return df[[KEY_COL, DIE_KEY_COL, value_col]].rename(columns={value_col: 'v'})

clf_die = {}
for name, path in clf_pool.items():
    clf_die[name] = {sp: _load_die(path, sp, 'prob') for sp in ['oof', 'val', 'test']}

reg_die = {}
for name, path in reg_pool.items():
    reg_die[name] = {sp: _load_die(path, sp, 'pred') for sp in ['oof', 'val', 'test']}

# ── 정합 검증: 같은 split 내에서 모든 clf/reg 가 동일한 (KEY, DIE_KEY) set + 길이 ──
ref_die = {}
for sp in ['oof', 'val', 'test']:
    ref = next(iter(clf_die.values()))[sp][[KEY_COL, DIE_KEY_COL]]
    ref_die[sp] = ref
    for name, d in {**clf_die, **reg_die}.items():
        cur = d[sp][[KEY_COL, DIE_KEY_COL]]
        if len(cur) != len(ref):
            raise ValueError(f'{sp}/{name}: row 수 {len(cur)} != ref {len(ref)}')
        if not (cur.values == ref.values).all():
            raise ValueError(f'{sp}/{name}: (KEY, DIE_KEY) 순서 불일치')

print('[정합 OK] 모든 clf/reg die-level csv 가 동일한 (KEY, DIE_KEY) 순서')
for sp in ['oof', 'val', 'test']:
    print(f'  {sp}: die={len(ref_die[sp]):,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[정합 OK] 모든 clf/reg die-level csv 가 동일한 (KEY, DIE_KEY) 순서
  oof: die=104,748
  val: die=34,908
  test: die=34,916


## 3. M × N grid 곱셈 + unit 집계 + RMSE

`final_die = prob × pred` → unit mean(KEY) 집계 → unit RMSE.

In [3]:
def _unit_mean(die_df, value_col='v'):
    return die_df.groupby(KEY_COL, sort=False)[value_col].mean()

def _rmse(p, y):
    p = p.loc[y.index]
    return float(np.sqrt(np.mean((p.values - y.values) ** 2)))

summary_rows = []
combined_unit = {}   # {(clf, reg): {'oof': series, 'val': series, 'test': series}}

for c_name in clf_pool:
    for r_name in reg_pool:
        unit_preds = {}
        for sp, y_true in [('oof', y_train), ('val', y_val), ('test', y_test)]:
            cdf = clf_die[c_name][sp]
            rdf = reg_die[r_name][sp]
            # 같은 순서 보장됨 (앞 셀 검증) → 직접 곱
            final_die = cdf['v'].values * rdf['v'].values
            tmp = pd.DataFrame({KEY_COL: cdf[KEY_COL].values, 'v': final_die})
            unit_preds[sp] = _unit_mean(tmp)

        combined_unit[(c_name, r_name)] = unit_preds
        summary_rows.append({
            'clf':  c_name,
            'reg':  r_name,
            'oof':  _rmse(unit_preds['oof'],  y_train),
            'val':  _rmse(unit_preds['val'],  y_val),
            'test': _rmse(unit_preds['test'], y_test),
        })

summary = pd.DataFrame(summary_rows).sort_values('val').reset_index(drop=True)
print('=== M × N Grid 결과 (val 오름차순) ===')
print(summary.to_string(index=False, float_format='%.6f'))

best_row = summary.iloc[0]
print(f'\nGrid best: clf={best_row["clf"]} × reg={best_row["reg"]} → val={best_row["val"]:.6f} test={best_row["test"]:.6f}')

=== M × N Grid 결과 (val 오름차순) ===
     clf      reg      oof      val     test
catboost catboost 0.008298 0.005739 0.008429
catboost     lgbm 0.008291 0.005741 0.008431
catboost      xgb 0.008290 0.005742 0.008431
     xgb       et 0.008328 0.005751 0.008433
     xgb     enet 0.008328 0.005752 0.008434
catboost       et 0.008307 0.005753 0.008421
catboost     enet 0.008308 0.005754 0.008422
     xgb catboost 0.008333 0.005771 0.008460
     xgb     lgbm 0.008327 0.005776 0.008464
     xgb      xgb 0.008325 0.005777 0.008464
    lgbm       et 0.008352 0.005786 0.008462
      et     lgbm 0.008299 0.005787 0.008469
    lgbm     enet 0.008353 0.005788 0.008463
      et      xgb 0.008299 0.005790 0.008471
      et catboost 0.008323 0.005805 0.008481
    lgbm catboost 0.008365 0.005818 0.008496
    lgbm     lgbm 0.008363 0.005825 0.008502
    lgbm      xgb 0.008361 0.005825 0.008502
      et       et 0.008402 0.005951 0.008566
      et     enet 0.008403 0.005953 0.008567

Grid best: clf=catboo

## 4. 모든 조합 unit csv 저장

In [4]:
for (c_name, r_name), preds in combined_unit.items():
    sub = os.path.join(OUT_DIR, f'{c_name}_x_{r_name}')
    os.makedirs(sub, exist_ok=True)
    for sp, y_true in [('oof', y_train), ('val', y_val), ('test', y_test)]:
        s = preds[sp]
        df = pd.DataFrame({
            KEY_COL:  s.index.values,
            'pred':   s.values,
            'health': y_true.reindex(s.index).values,
        })
        df.to_csv(os.path.join(sub, f'{sp}_unit.csv'), index=False)

summary.to_csv(os.path.join(OUT_DIR, 'grid_summary.csv'), index=False)
print(f'저장 완료: {OUT_DIR}')
print(f'  combined/{{clf}}_x_{{reg}}/(oof|val|test)_unit.csv  × {len(combined_unit)}개 조합')
print(f'  grid_summary.csv (val 오름차순)')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\two_stage_grid\combined
  combined/{clf}_x_{reg}/(oof|val|test)_unit.csv  × 20개 조합
  grid_summary.csv (val 오름차순)


## 5. Position weighted avg (grid 조합별 Optuna)

각 (clf, reg) grid 조합마다 die→unit 집계를 단순 mean 대신 **position 별 가중평균** 으로 변경. Optuna 가 4 weight (Dirichlet 정규화로 합=1) 를 OOF unit RMSE 최소화 방향 탐색.

- **8 grid 조합 × 50 trial = 400 trial 총량**
- **4 axis (w1, w2, w3, w4)** per study, Dirichlet 정규화 (`w_i = raw_i / Σraw`)
- **mean baseline 도 같이 측정** → val/test 모두 출력해서 OOF↔val gap 으로 overfit 모니터링
- 결과 csv: `combined/{clf}_x_{reg}/{oof,val,test}_unit_weighted.csv` (mean 버전과 별도)
- weighted_summary.csv 로 8 조합 결과 정렬

In [5]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from utils.data import load_all
xs_full, _ = load_all()
pos_map = xs_full.set_index(DIE_KEY_COL)['position']

def _add_pos(ref_df):
    df = ref_df.copy()
    df['position'] = df[DIE_KEY_COL].map(pos_map).astype(int)
    return df

ref_oof_pos  = _add_pos(ref_die['oof'])
ref_val_pos  = _add_pos(ref_die['val'])
ref_test_pos = _add_pos(ref_die['test'])

def _pivot_die(uid, pos, vals):
    df = pd.DataFrame({KEY_COL: uid, 'pos': pos, 'v': vals})
    return df.pivot(index=KEY_COL, columns='pos', values='v')

N_TRIALS_POS = 50
weighted_results = {}

print('=== Position weighted avg (grid 조합별 Optuna, 50 trial × 8 study = 400 total) ===\n')
for c_name in clf_pool:
    for r_name in reg_pool:
        # die-level final pred = clf_prob × reg_pred
        final_oof  = clf_die[c_name]['oof']['v'].values  * reg_die[r_name]['oof']['v'].values
        final_val  = clf_die[c_name]['val']['v'].values  * reg_die[r_name]['val']['v'].values
        final_test = clf_die[c_name]['test']['v'].values * reg_die[r_name]['test']['v'].values

        # position pivot (n_units × 4)
        pivot_oof  = _pivot_die(ref_oof_pos[KEY_COL].values,  ref_oof_pos['position'].values,  final_oof)
        pivot_val  = _pivot_die(ref_val_pos[KEY_COL].values,  ref_val_pos['position'].values,  final_val)
        pivot_test = _pivot_die(ref_test_pos[KEY_COL].values, ref_test_pos['position'].values, final_test)

        # mean baseline
        mean_oof_arr  = pivot_oof.mean(axis=1).reindex(y_train.index).values
        mean_val_arr  = pivot_val.mean(axis=1).reindex(y_val.index).values
        mean_test_arr = pivot_test.mean(axis=1).reindex(y_test.index).values
        baseline_oof  = float(np.sqrt(np.mean((mean_oof_arr  - y_train.values)**2)))
        baseline_val  = float(np.sqrt(np.mean((mean_val_arr  - y_val.values)**2)))
        baseline_test = float(np.sqrt(np.mean((mean_test_arr - y_test.values)**2)))

        # numpy array (Optuna 안에서 재사용 — 빠름)
        piv_oof_arr  = pivot_oof.reindex(y_train.index).values   # (n_train, 4)
        piv_val_arr  = pivot_val.reindex(y_val.index).values
        piv_test_arr = pivot_test.reindex(y_test.index).values
        y_tr_arr = y_train.values
        y_vl_arr = y_val.values
        y_te_arr = y_test.values

        def objective(trial, _po=piv_oof_arr, _yo=y_tr_arr):
            w_raw = np.array([trial.suggest_float(f'w{p}', 0.05, 1.0) for p in [1,2,3,4]])
            w = w_raw / w_raw.sum()
            wpred = (_po * w).sum(axis=1)
            return float(np.sqrt(np.mean((wpred - _yo)**2)))

        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=SEED),
        )
        study.optimize(objective, n_trials=N_TRIALS_POS, show_progress_bar=False)

        # best weights
        bw = np.array([study.best_trial.params[f'w{p}'] for p in [1,2,3,4]])
        bw = bw / bw.sum()

        # weighted RMSE 전 split
        w_oof  = (piv_oof_arr  * bw).sum(axis=1)
        w_val  = (piv_val_arr  * bw).sum(axis=1)
        w_test = (piv_test_arr * bw).sum(axis=1)
        wo_rmse  = float(np.sqrt(np.mean((w_oof  - y_tr_arr)**2)))
        wv_rmse  = float(np.sqrt(np.mean((w_val  - y_vl_arr)**2)))
        wt_rmse  = float(np.sqrt(np.mean((w_test - y_te_arr)**2)))

        # series 형태도 저장 (csv 출력용)
        w_oof_s  = pd.Series(w_oof,  index=y_train.index)
        w_val_s  = pd.Series(w_val,  index=y_val.index)
        w_test_s = pd.Series(w_test, index=y_test.index)

        weighted_results[(c_name, r_name)] = {
            'baseline':     {'oof': baseline_oof, 'val': baseline_val, 'test': baseline_test},
            'weighted':     {'oof': wo_rmse, 'val': wv_rmse, 'test': wt_rmse},
            'best_weights': bw.tolist(),
            'pred':         {'oof': w_oof_s, 'val': w_val_s, 'test': w_test_s},
        }

        # 출력
        print(f'  [{c_name} × {r_name}]')
        print(f'    weights: p1={bw[0]:.3f} p2={bw[1]:.3f} p3={bw[2]:.3f} p4={bw[3]:.3f}')
        print(f'    {"baseline (mean)":18s}  oof={baseline_oof:.6f}  val={baseline_val:.6f}  test={baseline_test:.6f}')
        print(f'    {"weighted":18s}  oof={wo_rmse:.6f}  val={wv_rmse:.6f}  test={wt_rmse:.6f}')
        print(f'    Δ (weighted-mean)   oof={wo_rmse-baseline_oof:+.6f} val={wv_rmse-baseline_val:+.6f} test={wt_rmse-baseline_test:+.6f}')
        if (wo_rmse < baseline_oof - 1e-5) and (wv_rmse > baseline_val + 5e-5):
            print(f'    ⚠ overfit 의심 (OOF 개선 / val 악화)')
        elif wv_rmse < baseline_val - 1e-5:
            print(f'    ✓ val 개선')
        else:
            print(f'    △ val 변동 미미')
        print()

# weighted summary csv 저장
wr_rows = []
for (c, r), data in weighted_results.items():
    bw_ = data['best_weights']
    wr_rows.append({
        'clf': c, 'reg': r,
        'mean_oof':     data['baseline']['oof'],
        'mean_val':     data['baseline']['val'],
        'mean_test':    data['baseline']['test'],
        'weighted_oof': data['weighted']['oof'],
        'weighted_val': data['weighted']['val'],
        'weighted_test':data['weighted']['test'],
        'delta_val':    data['weighted']['val'] - data['baseline']['val'],
        'delta_test':   data['weighted']['test'] - data['baseline']['test'],
        'w1': bw_[0], 'w2': bw_[1], 'w3': bw_[2], 'w4': bw_[3],
    })
weighted_summary = pd.DataFrame(wr_rows).sort_values('weighted_val').reset_index(drop=True)
weighted_summary.to_csv(os.path.join(OUT_DIR, 'weighted_summary.csv'), index=False)

print('=== Weighted summary (val 오름차순) ===')
print(weighted_summary.to_string(index=False, float_format='%.6f'))

# weighted unit csv 저장 (외부 stacking 에서 base 로 사용 가능)
for (c, r), data in weighted_results.items():
    sub = os.path.join(OUT_DIR, f'{c}_x_{r}')
    os.makedirs(sub, exist_ok=True)
    for sp_key, y_ref in [('oof', y_train), ('val', y_val), ('test', y_test)]:
        s = data['pred'][sp_key]
        df = pd.DataFrame({
            KEY_COL: s.index.values,
            'pred':  s.values,
            'health': y_ref.reindex(s.index).values,
        })
        df.to_csv(os.path.join(sub, f'{sp_key}_unit_weighted.csv'), index=False)
print(f'\nweighted unit csv 저장 완료 (8 grid × 3 split)')

Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
=== Position weighted avg (grid 조합별 Optuna, 50 trial × 8 study = 400 total) ===

  [catboost × catboost]
    weights: p1=0.233 p2=0.226 p3=0.281 p4=0.259
    baseline (mean)     oof=0.008298  val=0.005739  test=0.008429
    weighted            oof=0.008298  val=0.005739  test=0.008429
    Δ (weighted-mean)   oof=-0.000000 val=+0.000000 test=-0.000000
    △ val 변동 미미

  [catboost × enet]
    weights: p1=0.243 p2=0.258 p3=0.245 p4=0.254
    baseline (mean)     oof=0.008308  val=0.005754  test=0.008422
    weighted            oof=0.008308  val=0.005754  test=0.008422
    Δ (weighted-mean)   oof=-0.000000 val=-0.000000 test=+0.000000
    △ val 변동 미미

  [catboost × et]
    weights: p1=0.243 p2=0.258 p3=0.245 p4=0.254
    baseline (mean)     oof=0.008307  val=0.005753  test=0.008421
    weighted            oof=0.008307  val=0.005753  test=0.008421
    Δ (weighted-mean)   oof=-0.000000 val=-0.000000 test=-0.000000
    △ val 변동 미미


## 6. 메타 저장

In [6]:
meta = {
    'clf_pool':  list(clf_pool),
    'reg_pool':  list(reg_pool),
    'n_grid_combinations': len(combined_unit),
    'grid_summary_top10': summary.head(10).to_dict(orient='records'),
    'grid_best_mean': {
        'clf': str(best_row['clf']), 'reg': str(best_row['reg']),
        'oof': float(best_row['oof']), 'val': float(best_row['val']), 'test': float(best_row['test']),
    },
    'weighted_grid': {
        'n_trials_per_study': N_TRIALS_POS,
        'n_studies':          len(weighted_results),
        'summary_sorted':     weighted_summary.to_dict(orient='records'),
        'best_weighted_val':  float(weighted_summary.iloc[0]['weighted_val']),
        'best_combo':         f'{weighted_summary.iloc[0]["clf"]} × {weighted_summary.iloc[0]["reg"]}',
    },
}
with open(os.path.join(OUT_DIR, 'combine_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f_)
    if os.path.isfile(p):
        sz = os.path.getsize(p) / 1024
        print(f'  {f_:30s}  {sz:>10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\two_stage_grid\combined
  all_base_rmse.csv                      1.4 KB
  combine_meta.json                     13.4 KB
  final_stack_oof.csv                  970.0 KB
  final_stack_test.csv                 323.4 KB
  final_stack_val.csv                  323.5 KB
  grid_summary.csv                       1.5 KB
  stacking_coefs.csv                     1.0 KB
  weighted_summary.csv                   5.2 KB


## 7. 요약

In [7]:
print('=' * 80)
print(f' Two-Stage Grid Combine — 결과 요약')
print('=' * 80)
print(f'  CLF 모델: {len(clf_pool)} | REG 모델: {len(reg_pool)} | Grid 조합: {len(combined_unit)}')
print('-' * 80)
print(f'  [Mean baseline] Grid best (val): clf={best_row["clf"]}, reg={best_row["reg"]}')
print(f'    val={best_row["val"]:.6f}, test={best_row["test"]:.6f}')
print('-' * 80)
print(f'  [Position weighted avg] Top 3 (val 오름차순):')
for i, row in weighted_summary.head(3).iterrows():
    print(f'    {i+1}. {row["clf"]} × {row["reg"]:5s}  '
          f'val={row["weighted_val"]:.6f} (Δmean={row["delta_val"]:+.6f})  '
          f'test={row["weighted_test"]:.6f}  '
          f'w=[{row["w1"]:.2f},{row["w2"]:.2f},{row["w3"]:.2f},{row["w4"]:.2f}]')
print('-' * 80)
n_improved = (weighted_summary['delta_val'] < -1e-5).sum()
n_overfit  = ((weighted_summary['weighted_oof'] < weighted_summary['mean_oof'] - 1e-5) &
              (weighted_summary['delta_val'] > 5e-5)).sum()
print(f'  Position weighted: val 개선 {n_improved}/{len(weighted_summary)}, overfit 의심 {n_overfit}/{len(weighted_summary)}')
print('-' * 80)
print(f'  → mean / weighted unit csv 모두 저장 완료')
print(f'  → 외부 stacking 노트북에서 8 mean + 8 weighted 모두 base 로 사용 가능')
print('=' * 80)

 Two-Stage Grid Combine — 결과 요약
  CLF 모델: 4 | REG 모델: 5 | Grid 조합: 20
--------------------------------------------------------------------------------
  [Mean baseline] Grid best (val): clf=catboost, reg=catboost
    val=0.005739, test=0.008429
--------------------------------------------------------------------------------
  [Position weighted avg] Top 3 (val 오름차순):
    1. catboost × catboost  val=0.005739 (Δmean=+0.000000)  test=0.008429  w=[0.23,0.23,0.28,0.26]
    2. catboost × lgbm   val=0.005741 (Δmean=-0.000000)  test=0.008431  w=[0.22,0.24,0.27,0.27]
    3. catboost × xgb    val=0.005742 (Δmean=+0.000000)  test=0.008431  w=[0.25,0.24,0.24,0.27]
--------------------------------------------------------------------------------
  Position weighted: val 개선 0/20, overfit 의심 0/20
--------------------------------------------------------------------------------
  → mean / weighted unit csv 모두 저장 완료
  → 외부 stacking 노트북에서 8 mean + 8 weighted 모두 base 로 사용 가능
